# AI Herding · Next Step 2: Order lifecycle and resource logs

**Reuse the working directory from Step 1 and run the 8 code cells in order in Jupyter. No model API or API key is needed.**

Connect the following information:

**Complete decision → proposed orders → acceptance/rejection → quantity adjustment → execution/resting order/cancellation/replacement → resource release.**

- Assign stable `decision_id` values to successfully obtained decisions and retain their complete original order lists and Add/Cancel/Replace actions.
- Keep the engine's `order_id`, assign `trade_id` values, and record both counterparty order IDs.
- Save event order and available/committed cash and shares before and after each operation.
- Classify full, partial, and no execution using actual traded quantities; record cancellation separately.
- Reconcile logs, trades, and account commitments and check whether enabling the observer changes market outcomes.

**These are offline engine checks with manually specified responses.** They call the project's order creation, validation, matching, transfer, and cancellation services without calling an LLM. The 12 cases test the logging mechanism; they do not recover all missing links in the four historical runs or provide new LLM experimental findings.

Reuse Step 1's size cap and add a reusable observer in a new `source_logged` copy. The scope is a single stock, zero fees, and no dividend or interest payments. The test subprocess blocks network connections.

Example outputs were cleared from the distributed notebook. Run the cells in order and press `Ctrl + S` to retain the local results.


## Code cell 1/8: Locate the Step 1 working directory

Keep the generated `AI_Herding_Step1_Work` folder. Placing this notebook near the Step 1 notebook usually allows automatic discovery.

If the folder cannot be found, set `STEP1_WORK_OVERRIDE` to the working directory printed by Step 1, for example `r"C:\Users\YOUR_USERNAME\AI_Herding_Step1_Work"`.


In [ ]:
import sys, os, json, csv, hashlib, subprocess, shutil, ast
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict
from html import escape

STEP1_WORK_OVERRIDE = None  # If automatic discovery fails, set this to the raw-string path of the Step 1 working directory.
if sys.version_info < (3, 11):
    raise RuntimeError("Use a Python 3.11 or later notebook kernel.")

hint = STEP1_WORK_OVERRIDE or os.environ.get("AHE_STEP1_WORKDIR")
candidates = [Path(hint).expanduser()] if hint else []
if not hint:
    candidates += [Path.cwd() / "AI_Herding_Step1_Work", Path.home() / "AI_Herding_Step1_Work"]
    candidates += [p / "AI_Herding_Step1_Work" for p in list(Path.cwd().parents)[:3]]
STEP1_WORK = next((p.resolve() for p in candidates if (p / "source_capped/src/base_sim.py").is_file()), None)
if STEP1_WORK is None:
    raise FileNotFoundError("Step 1 working directory not found. Set STEP1_WORK_OVERRIDE to the AI_Herding_Step1_Work path printed by Step 1.")

CAPPED_ROOT = STEP1_WORK / "source_capped"
EXPECTED_CAP_SHA256 = {'src/market/orders/order_state_manager.py': '9ed4404e4c54f0dfd32c4bff1b047264fdaab19b2eb3a1c47b15f60679a28020', 'src/market/orders/order_service_factory.py': '3fa6ae07a599b36a9f095a207cb082b6681ad3817bf0d7f633e4959b428586de', 'src/base_sim.py': 'b5dc6d64012d372be01771ca90385db9a3da282e693c03e97ce2936ab28b379d'}
for relative_path, expected in EXPECTED_CAP_SHA256.items():
    text = (CAPPED_ROOT / relative_path).read_text(encoding="utf-8")
    if hashlib.sha256(text.encode("utf-8")).hexdigest() != expected:
        raise RuntimeError(f"Step 1 source differs from the verified version: {relative_path}. Check the source and keep local changes first.")

step1_summary = json.loads((STEP1_WORK / "results/step1_summary.json").read_text(encoding="utf-8"))
assert step1_summary["configured_limit"] == 1000
WORK_DIR = STEP1_WORK.parent / "AI_Herding_Step2_Work"
WORK_DIR.mkdir(parents=True, exist_ok=True)
LOGGED_ROOT = WORK_DIR / "source_logged"
TEST_PYTHON = STEP1_WORK / "test_environment" / ("Scripts/python.exe" if os.name == "nt" else "bin/python")
if not TEST_PYTHON.exists():
    raise FileNotFoundError("Step 1 test environment not found. Run code cell 3 of the Step 1 notebook first.")

print("Notebook Python：", sys.version.split()[0])
print("Step 1 directory found: ", STEP1_WORK)
print("Step 1 cap-source checks: PASS")
print("Working directory for this step: ", WORK_DIR)


## Code cell 2/8: Prepare display and test dependencies

Reuse the Step 1 test environment. Install NumPy, Pydantic, and Pandas only if missing. Installing dependencies may use the network but does not call a model API.


In [ ]:
probe_code = r"""
import sys, json
assert sys.version_info >= (3, 11)
needed=[]
for name, spec in [('numpy','numpy>=1.26'),('pandas','pandas>=2.2')]:
    try:
        __import__(name)
    except (ImportError, OSError):
        needed.append(spec)
try:
    import pydantic
    version=tuple(int(v) for v in pydantic.__version__.split('.')[:2])
    if version[0] != 2 or version < (2,11):
        needed.append('pydantic>=2.11,<3')
except (ImportError,OSError):
    needed.append('pydantic>=2.11,<3')
print(json.dumps(needed))
"""
probe = subprocess.run([str(TEST_PYTHON), "-c", probe_code], text=True, capture_output=True, check=True)
needed = json.loads(probe.stdout)
if needed:
    print("Installing offline test dependencies: ", ", ".join(needed))
    install = subprocess.run([str(TEST_PYTHON), "-m", "pip", "install", "--disable-pip-version-check", *needed],
                             text=True, capture_output=True)
    if install.returncode:
        raise RuntimeError("Dependency installation failed:\n" + install.stdout[-2000:] + install.stderr[-3000:])


def read_csv(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def read_jsonl(path):
    return [json.loads(line) for line in Path(path).read_text(encoding="utf-8").splitlines() if line.strip()]


def display_table(rows, columns):
    try:
        from IPython.display import display, HTML
        header = "".join(f"<th>{escape(str(label))}</th>" for key, label in columns)
        body = "".join("<tr>" + "".join(f"<td>{escape(str(row.get(key,'')))}</td>" for key, label in columns) + "</tr>" for row in rows)
        display(HTML("<table style='border-collapse:collapse' border='1' cellpadding='6'><thead><tr>" + header + "</tr></thead><tbody>" + body + "</tbody></table>"))
    except ImportError:
        for row in rows:
            print({label: row.get(key, "") for key, label in columns})

print("Test environment ready. No API key is required.")


## Code cell 3/8: Inspect missing linkage fields in historical logs

Inspect the four existing 15-round runs. Missing IDs in the old CSV files cannot be reconstructed without evidence. The new observer is intended for subsequent runs and is validated here with offline cases.


In [ ]:
legacy_rows = []
for run_id in ["fig6_above_baseline", "fig6_above_prompt_adjusted", "fig6_below_baseline", "fig6_below_prompt_adjusted"]:
    run = CAPPED_ROOT / "data/runs" / run_id
    order_path, trade_path = run / "data/order_data.csv", run / "data/trade_data.csv"
    with order_path.open(encoding="utf-8-sig", newline="") as stream:
        order_fields = set(csv.DictReader(stream).fieldnames or [])
    with trade_path.open(encoding="utf-8-sig", newline="") as stream:
        trade_fields = set(csv.DictReader(stream).fieldnames or [])
    legacy_rows.append({
        "run_id": run_id,
        "order_id": "Present" if "order_id" in order_fields else "Missing",
        "decision_id": "Present" if "decision_id" in order_fields else "Missing",
        "trade_order_ids": "Present" if {"buyer_order_id","seller_order_id"} <= trade_fields else "Missing",
    })
display_table(legacy_rows, [("run_id","Historical run"),("order_id","Order CSV: order_id"),
                            ("decision_id","Order CSV: decision_id"),("trade_order_ids","Trade CSV: both counterparty order IDs")])
print("Add complete logging and check linkage next. Do not fill missing historical fields with guesses.")


## Code cell 4/8: Create the instrumented test copy

This cell contains the complete observer and offline runner. Run the entire cell.

The observer wraps the project's actual methods and records their results without rewriting matching rules. It augments the `DataRecorder` order/trade records and separately saves complete decisions, the event stream, and round snapshots.

Keep **execution degree** (none/partial/full) separate from **terminal state** (still resting/cancelled/etc.). For example, retain both facts when an order is partially filled and its remainder is later cancelled.


In [ ]:
AUDIT_MODULE_SOURCE = r"""'''Observe real decision, order, resource and trade services without changing their rules.

Single-stock audit support for the AI Herding research branch. Attach before
collecting decisions. Persisted quantities distinguish acceptance-time resizing,
post-acceptance resizing, actual executions, and cancelled remainders.
'''
from __future__ import annotations

import csv
import json
import threading
import uuid
from collections import defaultdict
from contextlib import contextmanager
from datetime import datetime, timezone
from enum import Enum
from functools import wraps
from pathlib import Path

from market.orders.order import Order, OrderState


def json_default(value):
    if isinstance(value, Enum):
        return value.value
    if isinstance(value, (datetime, Path)):
        return str(value)
    if hasattr(value, 'model_dump'):
        return value.model_dump(mode='json')
    if hasattr(value, 'item'):
        return value.item()
    raise TypeError(f'Unsupported audit value: {type(value).__name__}')


def clean(value):
    return json.loads(json.dumps(value, default=json_default, ensure_ascii=False))


def write_csv(path, rows):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    keys = list(dict.fromkeys(key for row in rows for key in row))
    with path.open('w', encoding='utf-8-sig', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=keys or ['run_id'])
        writer.writeheader()
        for row in rows:
            writer.writerow({key: json.dumps(value, ensure_ascii=False, default=json_default)
                             if isinstance(value, (dict, list)) else value
                             for key, value in row.items()})


class ObservedOrder(Order):
    '''Preserve Order behavior and observe assignments on attached instances only.'''
    _observed_fields = {
        'quantity', 'remaining_quantity', 'filled_quantity', 'state', 'price',
        'current_cash_commitment', 'current_share_commitment',
    }

    def __setattr__(self, name, value):
        observer = self.__dict__.get('_lifecycle_observer')
        old = self.__dict__.get(name)
        existed = name in self.__dict__
        object.__setattr__(self, name, value)
        if observer is not None and existed and name in self._observed_fields and old != value:
            observer.attribute_changed(self, name, old, value)


class LifecycleAudit:
    def __init__(self, context, agent_repository, order_repository, output_dir, run_id=None):
        self.context = context
        self.agents = agent_repository
        self.orders = order_repository
        self.output_dir = Path(output_dir)
        self.output_dir.mkdir(parents=True, exist_ok=True)
        self.run_id = run_id or 'lifecycle_' + uuid.uuid4().hex
        self.lock = threading.RLock()
        self.local = threading.local()
        self.event_seq = 0
        self.id_counter = defaultdict(int)
        self.events = []
        self.decisions = []
        self.proposals = []
        self.meta = {}
        self.pending_decisions = {}
        self.trades = []
        self.rounds = []
        self.order_snapshots = []
        self.agent_snapshots = []
        self.bound_methods = []
        self.event_path = self.output_dir / 'order_events.jsonl'
        if self.event_path.exists():
            raise FileExistsError(f'Choose a fresh audit output directory: {self.output_dir}')

    def new_id(self, kind):
        with self.lock:
            self.id_counter[kind] += 1
            return f'{self.run_id}:{kind}:{self.id_counter[kind]:05d}'

    @contextmanager
    def operation(self, **updates):
        old = getattr(self.local, 'operation', {})
        self.local.operation = {**old, **updates}
        try:
            yield
        finally:
            self.local.operation = old

    def resource(self, agent_id, stock_id='DEFAULT_STOCK'):
        agent = self.agents.get_agent(agent_id)
        return {
            'available_cash': float(agent.available_cash),
            'committed_cash': float(agent.committed_cash),
            'dividend_cash': float(agent.dividend_cash),
            'total_cash': float(agent.total_cash),
            'available_shares': int(agent.positions.get(stock_id, 0)),
            'committed_shares': int(agent.committed_positions.get(stock_id, 0)),
            'borrowed_shares': int(agent.borrowed_positions.get(stock_id, 0)),
        }

    def all_resources(self):
        return {str(aid): self.resource(aid) for aid in self.agents.get_all_agent_ids()}

    def order_view(self, order):
        return {
            'order_id': str(order.order_id), 'agent_id': str(order.agent_id),
            'stock_id': order.stock_id, 'side': order.side, 'order_type': order.order_type,
            'quantity': order.quantity, 'remaining_quantity': order.remaining_quantity,
            'filled_quantity': order.filled_quantity, 'state': order.state.value,
            'price': order.price, 'cash_commitment': order.current_cash_commitment,
            'share_commitment': order.current_share_commitment,
        }

    def emit(self, event_type, order=None, agent_id=None, before=None, after=None, **details):
        op = dict(getattr(self.local, 'operation', {}))
        oid = str(order.order_id) if order is not None else op.get('order_id', '')
        meta = self.meta.get(oid, {})
        if agent_id is None and order is not None:
            agent_id = order.agent_id
        row = {
            'run_id': self.run_id, 'timestamp_utc': datetime.now(timezone.utc).isoformat(),
            'round_index': int(self.context.round_number),
            'market_round': int(self.context.round_number) + 1,
            'event_type': event_type, 'agent_id': '' if agent_id is None else str(agent_id),
            'order_id': oid, 'decision_id': meta.get('decision_id', op.get('decision_id', '')),
            'proposal_id': meta.get('proposal_id', op.get('proposal_id', '')),
            'trigger_decision_id': op.get('trigger_decision_id', ''),
            'trade_id': op.get('trade_id', ''), 'activity': op.get('activity', ''),
            'buyer_order_id': op.get('buyer_order_id', ''),
            'seller_order_id': op.get('seller_order_id', ''),
            'resource_before': before, 'resource_after': after,
            'order': self.order_view(order) if order is not None else None,
            'details': {**{key: value for key, value in op.items()
                          if key in ('transition_notes', 'conversion_reason')}, **details},
        }
        with self.lock:
            self.event_seq += 1
            row['event_seq'] = self.event_seq
            row = clean(row)
            self.events.append(row)
            with self.event_path.open('a', encoding='utf-8') as stream:
                stream.write(json.dumps(row, ensure_ascii=False) + '\n')
        return row

    def attribute_changed(self, order, field, old, new):
        meta = self.meta[str(order.order_id)]
        op = getattr(self.local, 'operation', {})
        if field == 'remaining_quantity' and meta['accepted_quantity'] is not None and op.get('activity') != 'trade_execution':
            meta['post_acceptance_removed_quantity'] += float(old) - float(new)
        event_type = {
            'state': 'STATE_CHANGED', 'quantity': 'ORDER_QUANTITY_CHANGED',
            'remaining_quantity': 'ORDER_REMAINING_CHANGED',
            'filled_quantity': 'ORDER_FILLED_COUNTER_CHANGED', 'price': 'ORDER_PRICE_CHANGED',
            'current_cash_commitment': 'ORDER_CASH_COMMITMENT_CHANGED',
            'current_share_commitment': 'ORDER_SHARE_COMMITMENT_CHANGED',
        }[field]
        self.emit(event_type, order, after=self.resource(order.agent_id, order.stock_id),
                  field=field, value_before=old, value_after=new,
                  is_execution=op.get('activity') == 'trade_execution')

    def bind(self, obj, name, wrapper):
        original = getattr(obj, name)
        self.bound_methods.append((obj, name, original))
        setattr(obj, name, wraps(original)(wrapper(original)))

    def current_decision(self, agent_id):
        return self.pending_decisions.get((int(self.context.round_number), str(agent_id)))

    def attach(self, decision_service, state_manager, trade_service, engine, recorder=None):
        if self.orders.orders:
            raise ValueError('Attach the audit before creating orders; old missing events cannot be recreated')

        def wrap_decision(original):
            def call(agent_id, market_state, history, round_number):
                before = self.resource(agent_id)
                response = original(agent_id, market_state, history, round_number)
                decision = {
                    'run_id': self.run_id, 'decision_id': self.new_id('decision'),
                    'agent_id': str(agent_id), 'round_index': int(round_number),
                    'market_round': int(round_number) + 1,
                    'replace_decision': response['replace_decision'],
                    'response': clean(response), 'cancelled_order_ids': [],
                    'resource_at_decision': before,
                }
                with self.lock:
                    self.decisions.append(decision)
                    self.pending_decisions[(int(round_number), str(agent_id))] = decision
                with self.operation(activity='decision_collection', decision_id=decision['decision_id']):
                    self.emit('DECISION_RECORDED', agent_id=agent_id, before=before,
                              replace_decision=decision['replace_decision'], response=decision['response'])
                return response
            return call
        self.bind(self.agents, 'get_agent_decision', wrap_decision)

        def wrap_create(original):
            def call(order_details, agent_id):
                decision = self.current_decision(agent_id)
                if decision is None:
                    raise ValueError('No recorded decision for this order creation')
                raw = clean(order_details)
                proposal = {
                    'run_id': self.run_id, 'decision_id': decision['decision_id'],
                    'proposal_id': self.new_id('proposal'), 'agent_id': str(agent_id),
                    'round_index': int(self.context.round_number),
                    'requested_quantity': raw['quantity'], 'order_id': '',
                    'replace_decision': decision['replace_decision'],
                    'replaces_order_ids': list(decision['cancelled_order_ids'])
                        if decision['replace_decision'] == 'Replace' else [],
                    'request': raw,
                }
                self.proposals.append(proposal)
                with self.operation(activity='order_creation', decision_id=decision['decision_id'],
                                    proposal_id=proposal['proposal_id'], proposal=proposal):
                    self.emit('ORDER_PROPOSED', agent_id=agent_id, before=self.resource(agent_id), request=raw)
                    try:
                        return original(order_details, agent_id)
                    except Exception as error:
                        proposal['creation_error'] = str(error)
                        self.emit('ORDER_CREATION_REJECTED', agent_id=agent_id, reason=str(error))
                        raise
            return call
        self.bind(decision_service, 'create_order', wrap_create)

        def wrap_register(original):
            def call(order):
                proposal = getattr(self.local, 'operation', {}).get('proposal')
                oid = str(order.order_id)
                if proposal is None:
                    # Broker-created orders are explicitly separate from agent decisions.
                    proposal = {'decision_id': '', 'proposal_id': self.new_id('engine_proposal'),
                                'requested_quantity': order.quantity, 'replaces_order_ids': []}
                proposal['order_id'] = oid
                self.meta[oid] = {
                    'decision_id': proposal['decision_id'], 'proposal_id': proposal['proposal_id'],
                    'requested_quantity': proposal['requested_quantity'], 'accepted_quantity': None,
                    'post_acceptance_removed_quantity': 0.0, 'accepted': False,
                    'validation_reason': '', 'cancellation_reason': '',
                    'replaces_order_ids': list(proposal['replaces_order_ids']),
                }
                order.__class__ = ObservedOrder
                object.__setattr__(order, '_lifecycle_observer', self)
                object.__setattr__(order, 'decision_id', proposal['decision_id'])
                object.__setattr__(order, 'proposal_id', proposal['proposal_id'])
                result = original(order)
                self.emit('ORDER_REGISTERED', order, after=self.resource(order.agent_id, order.stock_id),
                          replaces_order_ids=proposal['replaces_order_ids'])
                return result
            return call
        self.bind(self.orders, 'create_order', wrap_register)

        def wrap_transition(original):
            def call(order_id, new_state, *args, **kwargs):
                notes = kwargs.get('notes', args[2] if len(args) > 2 else '')
                with self.operation(order_id=str(order_id), transition_notes=notes or ''):
                    return original(order_id, new_state, *args, **kwargs)
            return call
        self.bind(self.orders, 'transition_state', wrap_transition)

        def wrap_validate(original):
            def call(order, current_price):
                before = self.resource(order.agent_id, order.stock_id)
                with self.operation(activity='initial_validation', order_id=str(order.order_id)):
                    self.emit('VALIDATION_STARTED', order, before=before)
                    valid, reason = original(order, current_price)
                    after = self.resource(order.agent_id, order.stock_id)
                    meta = self.meta[str(order.order_id)]
                    if meta['accepted_quantity'] is None:
                        meta['accepted'] = bool(valid)
                        meta['accepted_quantity'] = float(order.quantity) if valid else 0.0
                        meta['validation_reason'] = reason
                        meta['validation_resource_before'] = before
                        meta['validation_resource_after'] = after
                    self.emit('VALIDATION_ACCEPTED' if valid else 'VALIDATION_REJECTED',
                              order, before=before, after=after, reason=reason,
                              requested_quantity=meta['requested_quantity'],
                              accepted_quantity=meta['accepted_quantity'])
                    return valid, reason
            return call
        self.bind(state_manager, 'handle_new_order', wrap_validate)

        def resource_order(agent_id):
            op = getattr(self.local, 'operation', {})
            if op.get('trade_id'):
                oid = op['buyer_order_id'] if str(agent_id) == op.get('buyer_id') else op['seller_order_id']
            else:
                oid = op.get('order_id')
            return self.orders.orders.get(oid)

        def wrap_resource(method_name, label):
            def factory(original):
                def call(agent_id, *args, **kwargs):
                    stock = kwargs.get('stock_id', 'DEFAULT_STOCK')
                    before = self.resource(agent_id, stock)
                    order = resource_order(agent_id)
                    self.emit(label + '_REQUESTED', order, agent_id, before=before, request=clean(kwargs))
                    result = original(agent_id, *args, **kwargs)
                    success_event = 'RESOURCE_COMMITTED' if label == 'RESOURCE_COMMIT' else 'RESOURCE_RELEASED'
                    self.emit(success_event if result.success else label + '_REJECTED', order, agent_id,
                              before=before, after=self.resource(agent_id, stock),
                              success=bool(result.success), message=result.message,
                              partial_allocation=bool(getattr(result, 'partial_fill', False)),
                              requested_amount=getattr(result, 'requested_amount', None),
                              committed_amount=getattr(result, 'committed_amount', None))
                    return result
                return call
            self.bind(self.agents, method_name, factory)
        wrap_resource('commit_resources', 'RESOURCE_COMMIT')
        wrap_resource('release_resources', 'RESOURCE_RELEASE')

        def wrap_position(original):
            def call(agent_id, changes):
                order = resource_order(agent_id)
                stock = changes.stock_id
                before = self.resource(agent_id, stock)
                result = original(agent_id, changes)
                self.emit('POSITION_UPDATED', order, agent_id, before=before,
                          after=self.resource(agent_id, stock),
                          cash_change=changes.cash_change, share_change=changes.shares_change)
                return result
            return call
        self.bind(self.agents, 'update_agent_position_after_trade', wrap_position)

        def wrap_cancel(original):
            def call(order, message='Cancelled', skip_sync=False):
                decision = self.current_decision(order.agent_id)
                trigger = decision['decision_id'] if decision else ''
                before = self.resource(order.agent_id, order.stock_id)
                with self.operation(activity='cancellation', order_id=str(order.order_id),
                                    trigger_decision_id=trigger):
                    self.emit('CANCEL_STARTED', order, before=before, reason=message)
                    result = original(order, message, skip_sync=skip_sync)
                    self.meta[str(order.order_id)]['cancellation_reason'] = message
                    if decision is not None:
                        decision['cancelled_order_ids'].append(str(order.order_id))
                    self.emit('CANCEL_COMPLETED', order, before=before,
                              after=self.resource(order.agent_id, order.stock_id), reason=message)
                    return result
            return call
        self.bind(state_manager, 'handle_single_order_cancellation', wrap_cancel)

        def wrap_trade(original):
            def call(trade):
                tid = self.new_id('trade')
                object.__setattr__(trade, 'trade_id', tid)
                before = {str(trade.buyer_id): self.resource(trade.buyer_id, trade.stock_id),
                          str(trade.seller_id): self.resource(trade.seller_id, trade.stock_id)}
                with self.operation(activity='trade_execution', trade_id=tid,
                                    order_id=str(trade.buyer_order_id),
                                    buyer_id=str(trade.buyer_id), seller_id=str(trade.seller_id),
                                    buyer_order_id=str(trade.buyer_order_id),
                                    seller_order_id=str(trade.seller_order_id)):
                    self.emit('TRADE_EXECUTION_STARTED', before=before,
                              quantity=trade.quantity, price=trade.price)
                    success = original(trade)
                    after = {str(trade.buyer_id): self.resource(trade.buyer_id, trade.stock_id),
                             str(trade.seller_id): self.resource(trade.seller_id, trade.stock_id)}
                    event = self.emit('TRADE_EXECUTED' if success else 'TRADE_REJECTED',
                                      before=before, after=after,
                                      quantity=trade.quantity, price=trade.price)
                    if success:
                        self.trades.append({
                            'run_id': self.run_id, 'trade_id': tid, 'event_seq': event['event_seq'],
                            'round_index': int(self.context.round_number),
                            'market_round': int(self.context.round_number) + 1,
                            'buyer_id': str(trade.buyer_id), 'seller_id': str(trade.seller_id),
                            'buyer_order_id': str(trade.buyer_order_id),
                            'seller_order_id': str(trade.seller_order_id),
                            'buyer_decision_id': self.meta[str(trade.buyer_order_id)]['decision_id'],
                            'seller_decision_id': self.meta[str(trade.seller_order_id)]['decision_id'],
                            'quantity': trade.quantity, 'price': trade.price, 'stock_id': trade.stock_id,
                        })
                    return success
            return call
        self.bind(trade_service, 'execute_trade', wrap_trade)

        def quote():
            return {'best_bid': engine.order_book.get_best_bid(),
                    'best_ask': engine.order_book.get_best_ask(),
                    'midpoint': engine.order_book.get_midpoint()}

        def wrap_match(original):
            def call(new_orders, current_price, round_number):
                before = quote()
                with self.operation(activity='matching'):
                    self.emit('MATCHING_STARTED', quotes=before,
                              new_order_ids=[str(o.order_id) for o in new_orders])
                    result = original(new_orders, current_price, round_number)
                    actual_ids = {t['trade_id'] for t in self.trades if t['round_index'] == int(round_number)}
                    returned_ids = {getattr(t, 'trade_id', '') for t in result.trades}
                    if actual_ids != returned_ids:
                        raise AssertionError('Returned trades differ from successfully executed trades')
                    after = quote()
                    self.emit('MATCHING_COMPLETED', quotes=after, volume=result.volume,
                              trade_ids=sorted(returned_ids), recorded_price=result.price)
                    self.rounds.append({
                        'run_id': self.run_id, 'round_index': int(round_number),
                        'market_round': int(round_number) + 1, 'recorded_price': result.price,
                        'trade_count': len(result.trades), 'trade_volume': result.volume,
                        'last_execution_this_round': result.trades[-1].price if result.trades else None,
                        **{'pre_' + key: value for key, value in before.items()},
                        **{'post_' + key: value for key, value in after.items()},
                    })
                    return result
            return call
        self.bind(engine, 'match_orders', wrap_match)

        def wrap_processing(original):
            def call(orders):
                result = original(orders)
                self.emit('ORDER_PROCESSING_SEQUENCE',
                          market_order_ids=[str(o.order_id) for o in result[0]],
                          limit_order_ids=[str(o.order_id) for o in result[1]])
                return result
            return call
        self.bind(engine.order_processing_service, 'split_orders_by_type', wrap_processing)

        def wrap_conversion(original):
            def call(order, current_price):
                with self.operation(activity='conversion', order_id=str(order.order_id),
                                    conversion_reason='Unfilled market order converted at an aggressive quote'):
                    self.emit('CONVERSION_STARTED', order)
                    result = original(order, current_price)
                    self.emit('CONVERSION_COMPLETED', order, success=result.success, reason=result.message)
                    return result
            return call
        self.bind(engine.market_order_handler._conversion_service, 'convert_to_aggressive_limit', wrap_conversion)

        if recorder is not None:
            self.attach_recorder(recorder)
        return self

    def attach_recorder(self, recorder):
        def wrap_order_records(original):
            def call(round_number, orders, timestamp):
                start = len(recorder.order_data)
                result = original(round_number, orders, timestamp)
                for record, order in zip(recorder.order_data[start:], orders):
                    meta = self.meta[str(order.order_id)]
                    record.update({
                        'order_id': str(order.order_id), 'decision_id': meta['decision_id'],
                        'proposal_id': meta['proposal_id'],
                        'requested_quantity': meta['requested_quantity'],
                        'accepted_quantity': meta['accepted_quantity'],
                        'remaining_quantity': order.remaining_quantity,
                        'filled_quantity': order.filled_quantity, 'order_state': order.state.value,
                    })
                return result
            return call
        self.bind(recorder, '_record_order_data', wrap_order_records)

        def wrap_trade_records(original):
            def call(round_number, trades, timestamp):
                start = len(recorder.trade_data)
                result = original(round_number, trades, timestamp)
                for record, trade in zip(recorder.trade_data[start:], trades):
                    record.update({
                        'trade_id': trade.trade_id,
                        'buyer_order_id': str(trade.buyer_order_id),
                        'seller_order_id': str(trade.seller_order_id),
                    })
                return result
            return call
        self.bind(recorder, '_record_trade_data', wrap_trade_records)

    def snapshot(self, label='round_end'):
        self.emit('SNAPSHOT', label=label)
        for order in self.orders.orders.values():
            self.order_snapshots.append({'run_id': self.run_id, 'round_index': int(self.context.round_number),
                                         'label': label, **self.order_view(order)})
        for agent_id, resource in self.all_resources().items():
            self.agent_snapshots.append({'run_id': self.run_id, 'round_index': int(self.context.round_number),
                                         'label': label, 'agent_id': agent_id, **resource})

    def lifecycle_rows(self):
        executed = defaultdict(float)
        for trade in self.trades:
            executed[trade['buyer_order_id']] += float(trade['quantity'])
            executed[trade['seller_order_id']] += float(trade['quantity'])
        rows = []
        for oid, order in self.orders.orders.items():
            oid = str(oid)
            meta = self.meta[oid]
            accepted = bool(meta['accepted'])
            accepted_qty = float(meta['accepted_quantity'] or 0)
            filled = executed[oid]
            remaining = float(order.remaining_quantity)
            removed = float(meta['post_acceptance_removed_quantity'])
            terminal = order.state in (OrderState.CANCELLED, OrderState.FILLED)
            if not accepted:
                status = 'rejected'
            elif filled == 0:
                status = 'unfilled'
            elif abs(filled - accepted_qty) < 1e-8:
                status = 'fully_filled'
            else:
                status = 'partially_filled'
            residual = accepted_qty - filled - remaining - removed if accepted else None
            rows.append({
                'run_id': self.run_id, 'order_id': oid, 'decision_id': meta['decision_id'],
                'proposal_id': meta['proposal_id'], 'agent_id': str(order.agent_id),
                'side': order.side, 'order_type': order.order_type,
                'requested_quantity': meta['requested_quantity'], 'accepted': accepted,
                'accepted_quantity': accepted_qty, 'executed_quantity': filled,
                'unaccepted_quantity_reduction': float(meta['requested_quantity']) - accepted_qty if accepted else None,
                'post_acceptance_removed_quantity': removed,
                'remaining_quantity_raw': remaining,
                'remaining_open_quantity': remaining if accepted and not terminal else 0,
                'cancelled_remaining_quantity': remaining if accepted and order.state == OrderState.CANCELLED else 0,
                'execution_status': status, 'final_state': order.state.value,
                'validation_reason': meta['validation_reason'],
                'cancellation_reason': meta['cancellation_reason'],
                'replaces_order_ids': meta['replaces_order_ids'],
                'cash_commitment_final': order.current_cash_commitment,
                'share_commitment_final': order.current_share_commitment,
                'quantity_ledger_residual': residual,
                'quantity_ledger_ok': residual is None or abs(residual) < 1e-8,
                'terminal_commitments_released': (abs(order.current_cash_commitment) < 1e-8 and
                                                  abs(order.current_share_commitment) < 1e-8) if terminal else None,
            })
        return rows

    def validate(self):
        rows = self.lifecycle_rows()
        assert [row['event_seq'] for row in self.events] == list(range(1, len(self.events) + 1))
        decision_ids = {row['decision_id'] for row in self.decisions}
        assert len(decision_ids) == len(self.decisions)
        assert len({row['trade_id'] for row in self.trades}) == len(self.trades)
        for row in rows:
            assert row['decision_id'] in decision_ids
            assert row['quantity_ledger_ok'], row
            order = self.orders.get_order(row['order_id'])
            assert abs(float(order.filled_quantity) - row['executed_quantity']) < 1e-8, row
            if row['terminal_commitments_released'] is not None:
                assert row['terminal_commitments_released'], row
        by_id = {row['order_id']: row for row in rows}
        for trade in self.trades:
            assert by_id[trade['buyer_order_id']]['accepted']
            assert by_id[trade['seller_order_id']]['accepted']
        total_legs = sum(row['executed_quantity'] for row in rows)
        assert abs(total_legs - 2 * sum(float(t['quantity']) for t in self.trades)) < 1e-8
        for aid in self.agents.get_all_agent_ids():
            orders = self.orders.get_agent_orders(aid)
            resource = self.resource(aid)
            assert abs(sum(o.current_cash_commitment for o in orders) - resource['committed_cash']) < 1e-6
            assert abs(sum(o.current_share_commitment for o in orders) - resource['committed_shares']) < 1e-6
        return rows

    def export(self, recorder=None):
        rows = self.validate()
        flat = []
        for event in self.events:
            row = {key: value for key, value in event.items()
                   if key not in ('resource_before', 'resource_after', 'order', 'details')}
            for label in ('before', 'after'):
                resource = event['resource_' + label]
                if isinstance(resource, dict) and 'available_cash' in resource:
                    row.update({label + '_' + key: value for key, value in resource.items()})
            row['order_snapshot'] = event['order']
            row['details'] = event['details']
            row['resource_before_json'] = event['resource_before']
            row['resource_after_json'] = event['resource_after']
            flat.append(row)
        write_csv(self.output_dir / 'order_events.csv', flat)
        write_csv(self.output_dir / 'decision_orders.csv', self.proposals)
        write_csv(self.output_dir / 'order_lifecycle.csv', rows)
        write_csv(self.output_dir / 'trades_linked.csv', self.trades)
        write_csv(self.output_dir / 'order_snapshots.csv', self.order_snapshots)
        write_csv(self.output_dir / 'agent_snapshots.csv', self.agent_snapshots)
        write_csv(self.output_dir / 'rounds.csv', self.rounds)
        with (self.output_dir / 'decisions.jsonl').open('w', encoding='utf-8') as stream:
            for decision in self.decisions:
                stream.write(json.dumps(clean(decision), ensure_ascii=False) + '\n')
        if recorder is not None:
            write_csv(self.output_dir / 'recorded_orders.csv', recorder.order_data)
            write_csv(self.output_dir / 'recorded_trades.csv', recorder.trade_data)
        return rows


def attach_to_simulation(simulation, output_dir, run_id=None):
    '''Attach before simulation.run(); this function itself makes no model calls.

    Calling simulation.run() with LLM agents can make model calls. The notebook
    instead uses manual-response agents and blocks network access in the worker.
    '''
    if simulation.is_multi_stock:
        raise NotImplementedError('This research logger currently targets the single-stock Figure 6-7 setting')
    audit = LifecycleAudit(simulation.context, simulation.agent_repository,
                           simulation.order_repository, output_dir, run_id)
    audit.attach(simulation.decision_service, simulation.order_state_manager,
                 simulation.trade_execution_service, simulation.matching_engine,
                 recorder=simulation.data_recorder)
    simulation.register_before_round(lambda sim, r: audit.emit('ROUND_STARTED'))
    simulation.register_after_round(lambda sim, r: (audit.snapshot(), audit.export(sim.data_recorder)))
    return audit
"""

OFFLINE_RUNNER_SOURCE = r"""'''Run manual-response lifecycle cases through the real market engine; no model API.'''
from __future__ import annotations
import argparse
import csv
import json
import logging
import random
import socket
import sys
from collections import Counter
from copy import deepcopy
from datetime import datetime, timezone
from pathlib import Path

parser = argparse.ArgumentParser()
parser.add_argument('--source', type=Path, required=True)
parser.add_argument('--output', type=Path, required=True)
args = parser.parse_args()
source = args.source.resolve()
output = args.output.resolve()
output.mkdir(parents=True, exist_ok=True)
sys.dont_write_bytecode = True
sys.path.insert(0, str(source / 'tests'))
import _logging_stub
_logging_stub.install()
logging.disable(logging.CRITICAL)
network_attempts = []
def deny_network(*unused_args, **unused_kwargs):
    network_attempts.append('blocked')
    raise RuntimeError('Network access is disabled for the lifecycle worker')
socket.socket.connect = deny_network
socket.socket.connect_ex = deny_network
socket.create_connection = deny_network

from agents.base_agent import BaseAgent
from agents.agent_manager.agent_repository import AgentRepository
from agents.agent_manager.base_agent_manager import AgentManager
from agents.agent_manager.services.agent_decision_service import AgentDecisionService
from agents.agent_manager.services.borrowing_repository import BorrowingRepository
from market.orders.order_repository import OrderRepository
from market.orders.order_book import OrderBook
from market.orders.order_service_factory import OrderServiceFactory
from market.state.sim_context import SimulationContext
from market.engine.match_engine import MatchingEngine
from market.data_recorder import DataRecorder
from services.shared_service_factory import SharedServiceFactory
from order_lifecycle_audit import LifecycleAudit, write_csv, clean


def order(side, quantity, price=15, kind='limit'):
    return {'decision': side.title(), 'quantity': quantity, 'order_type': kind,
            'price_limit': price if kind == 'limit' else None}


def decision(*orders, action='Add'):
    return {'orders': list(orders), 'replace_decision': action,
            'reasoning': 'Hand-written offline test fixture', 'response_source': 'manual_fixture'}


CASES = [
    {'id': 'accepted_unfilled', 'title': 'Bid 14 / ask 15: accepted but unfilled',
     'schedule': [{'buyer': decision(order('buy', 100, 14)), 'seller': decision(order('sell', 100, 15))}],
     'trades': 0, 'volume': 0},
    {'id': 'fully_filled', 'title': 'Crossed limits: both sides fully filled',
     'schedule': [{'buyer': decision(order('buy', 100)), 'seller': decision(order('sell', 100))}],
     'trades': 1, 'volume': 100},
    {'id': 'partially_filled', 'title': 'Buy 100 / sell 40: partial buy execution',
     'schedule': [{'buyer': decision(order('buy', 100)), 'seller': decision(order('sell', 40))}],
     'trades': 1, 'volume': 40},
    {'id': 'cancel_unfilled', 'title': 'Cancel a resting order: release commitments',
     'schedule': [{'buyer': decision(order('buy', 100))}, {'buyer': decision(action='Cancel')}],
     'trades': 0, 'volume': 0},
    {'id': 'replace_unfilled', 'title': 'Replace: release the old order, then submit the new order',
     'schedule': [{'buyer': decision(order('buy', 100, 14))},
                  {'buyer': decision(order('buy', 80, 13), action='Replace')}],
     'trades': 0, 'volume': 0},
    {'id': 'reject_cash', 'title': 'Insufficient cash: reject a buy order',
     'schedule': [{'buyer': decision(order('buy', 100))}],
     'params': {'buyer': {'cash': 10}}, 'trades': 0, 'volume': 0},
    {'id': 'reject_shares', 'title': 'Insufficient inventory and no borrowable shares: reject a sell order',
     'schedule': [{'seller': decision(order('sell', 100))}],
     'params': {'seller': {'shares': 0}}, 'trades': 0, 'volume': 0},
    {'id': 'reject_size', 'title': 'Inherited from Step 1: reject a 1001-share order',
     'schedule': [{'buyer': decision(order('buy', 1001))}], 'trades': 0, 'volume': 0},
    {'id': 'partial_then_cancel', 'title': 'Execute 40, then cancel the remaining 60',
     'schedule': [{'buyer': decision(order('buy', 100)), 'seller': decision(order('sell', 40))},
                  {'buyer': decision(action='Cancel')}], 'trades': 1, 'volume': 40},
    {'id': 'allocation_reduction', 'title': 'Partial borrow approval: resizing is not execution',
     'schedule': [{'seller': decision(order('sell', 1000))}],
     'params': {'seller': {'shares': 300, 'short': True}},
     'borrow_pool': 200, 'partial_borrows': True, 'trades': 0, 'volume': 0},
    {'id': 'market_resize', 'title': 'After partial market-order execution: adjust the remaining quantity',
     'schedule': [{'buyer': decision(order('buy', 1000, kind='market')),
                   'seller': decision(order('sell', 500, 10))}], 'trades': 1, 'volume': 500},
    {'id': 'later_round_fill', 'title': 'Rest in round 1, execute in round 2: cross-round linkage',
     'schedule': [{'buyer': decision(order('buy', 100, 14))},
                  {'seller': decision(order('sell', 100, 14))}], 'trades': 1, 'volume': 100},
]


class ManualFixtureAgent(BaseAgent):
    def make_decision(self, market_state, history, round_number):
        self.fixture_calls += 1
        return deepcopy(self.responses.get(round_number, decision()))


def normalized(orepo, agent_repo, trades, prices):
    return clean({
        'orders': [dict(agent_id=str(o.agent_id), quantity=o.quantity,
                        remaining=o.remaining_quantity, filled=o.filled_quantity,
                        state=o.state.value, price=o.price,
                        cash_commitment=o.current_cash_commitment,
                        share_commitment=o.current_share_commitment)
                   for o in orepo.orders.values()],
        'agents': [dict(agent_id=str(a.agent_id), cash=a.cash,
                        committed_cash=a.committed_cash, positions=a.positions,
                        committed_positions=a.committed_positions, borrowed_positions=a.borrowed_positions)
                   for a in agent_repo.get_all_agents()],
        'trades': [dict(buyer=str(t.buyer_id), seller=str(t.seller_id),
                       quantity=t.quantity, price=t.price, round=t.round) for t in trades],
        'prices': prices,
    })


def run_case(case, observed):
    random.seed(42)
    logger = logging.getLogger('step2')
    context = SimulationContext(len(case['schedule']), 15.0, 28.0, 28.0, 0.0)
    agents = []
    for aid in ('buyer', 'seller'):
        config = case.get('params', {}).get(aid, {})
        agent = ManualFixtureAgent(aid, initial_cash=config.get('cash', 100000),
                                   initial_shares=config.get('shares', 1000),
                                   allow_short_selling=config.get('short', False), initial_price=15.0)
        agent.responses = {r: choices.get(aid, decision()) for r, choices in enumerate(case['schedule'])}
        agent.fixture_calls = 0
        agents.append(agent)
    borrow = BorrowingRepository(case.get('borrow_pool', 0), case.get('partial_borrows', False), logger)
    agent_repo = AgentRepository(agents, logger, context, borrowing_repository=borrow)
    orepo = OrderRepository(logger)
    book = OrderBook(context, orepo, logger)
    SharedServiceFactory.reset()
    SharedServiceFactory.initialize(order_book=book, transaction_cost=0.0)
    state, execution = OrderServiceFactory.create_services(orepo, agent_repo, book, logger, max_order_size=1000)
    manager = AgentManager(agent_repo, context, None, logger, logger, orepo, state, book,
                           commitment_calculator=SharedServiceFactory.get_commitment_calculator(),
                           position_calculator=SharedServiceFactory.get_position_calculator())
    decisions = AgentDecisionService(agent_repo, orepo, state, logger, logger, context, max_concurrency=8)
    engine = MatchingEngine(book, manager, agent_repo, orepo, context, state, logger, logger, execution)
    recorder = object.__new__(DataRecorder)
    recorder.order_data, recorder.trade_data = [], []
    audit = None
    if observed:
        audit = LifecycleAudit(context, agent_repo, orepo, output / case['id'],
                               run_id=output.name + ':' + case['id'])
        audit.attach(decisions, state, execution, engine, recorder)
    initial_cash = sum(a.total_cash for a in agents)
    initial_net_shares = sum(sum(a.positions.values()) + sum(a.committed_positions.values())
                             - sum(a.borrowed_positions.values()) for a in agents)
    trades, prices = [], []
    for round_index in range(len(case['schedule'])):
        context.round_number = round_index
        market_state = {'price': context.current_price, 'best_bid': book.get_best_bid(),
                        'best_ask': book.get_best_ask()}
        new_orders = decisions.collect_decisions(market_state, [], round_index)
        result = engine.match_orders(new_orders, context.current_price, round_index)
        context.current_price = result.price
        trades.extend(result.trades)
        prices.append(result.price)
        timestamp = datetime.now(timezone.utc).isoformat()
        recorder._record_order_data(round_index, new_orders, timestamp)
        recorder._record_trade_data(round_index, result.trades, timestamp)
        if audit:
            audit.snapshot()
            audit.validate()
    assert len(trades) == case['trades'], (case['id'], 'trade_count')
    assert sum(t.quantity for t in trades) == case['volume'], (case['id'], 'trade_volume')
    assert abs(sum(a.total_cash for a in agents) - initial_cash) < 1e-6
    final_net_shares = sum(sum(a.positions.values()) + sum(a.committed_positions.values())
                           - sum(a.borrowed_positions.values()) for a in agents)
    assert abs(final_net_shares - initial_net_shares) < 1e-6
    rows = audit.export(recorder) if audit else []
    if audit:
        buyer = [row for row in rows if row['agent_id'] == 'buyer']
        seller = [row for row in rows if row['agent_id'] == 'seller']
        name = case['id']
        if name == 'accepted_unfilled':
            assert all(r['accepted'] and r['execution_status'] == 'unfilled' for r in rows)
        elif name == 'fully_filled':
            assert all(r['execution_status'] == 'fully_filled' for r in rows)
        elif name == 'partially_filled':
            assert buyer[0]['executed_quantity'] == 40 and buyer[0]['remaining_open_quantity'] == 60
            assert buyer[0]['execution_status'] == 'partially_filled'
        elif name == 'cancel_unfilled':
            assert buyer[0]['execution_status'] == 'unfilled' and buyer[0]['final_state'] == 'cancelled'
        elif name == 'replace_unfilled':
            assert len(buyer) == 2 and buyer[0]['order_id'] in buyer[1]['replaces_order_ids']
            released = [e['event_seq'] for e in audit.events if e['event_type'] == 'RESOURCE_RELEASED'
                        and e['order_id'] == buyer[0]['order_id']]
            committed = [e['event_seq'] for e in audit.events if e['event_type'] == 'RESOURCE_COMMITTED'
                         and e['order_id'] == buyer[1]['order_id']]
            assert released and committed and max(released) < min(committed)
        elif name.startswith('reject_'):
            assert len(rows) == 1 and not rows[0]['accepted'] and rows[0]['execution_status'] == 'rejected'
            meta = audit.meta[rows[0]['order_id']]
            assert meta['validation_resource_before'] == meta['validation_resource_after']
        elif name == 'partial_then_cancel':
            assert buyer[0]['execution_status'] == 'partially_filled' and buyer[0]['final_state'] == 'cancelled'
            assert buyer[0]['cancelled_remaining_quantity'] == 60
        elif name == 'allocation_reduction':
            assert seller[0]['requested_quantity'] == 1000 and seller[0]['accepted_quantity'] == 500
            assert seller[0]['execution_status'] == 'unfilled' and seller[0]['executed_quantity'] == 0
        elif name == 'market_resize':
            assert buyer[0]['accepted_quantity'] == 1000 and buyer[0]['executed_quantity'] == 500
            assert buyer[0]['remaining_open_quantity'] == 454
            assert buyer[0]['post_acceptance_removed_quantity'] == 46
        elif name == 'later_round_fill':
            assert all(r['execution_status'] == 'fully_filled' for r in rows)
            bid = buyer[0]['order_id']
            assert any(s['order_id'] == bid and s['round_index'] == 0 and s['remaining_quantity'] == 100
                       for s in audit.order_snapshots)
            assert audit.trades[0]['round_index'] == 1
        accepted_ids = {r['order_id'] for r in rows if r['accepted']}
        assert {r['order_id'] for r in recorder.order_data} == accepted_ids
        assert {r['trade_id'] for r in recorder.trade_data} == {t['trade_id'] for t in audit.trades}
    return {
        'normalized': normalized(orepo, agent_repo, trades, prices), 'rows': rows,
        'audit': audit, 'recorder': recorder,
        'manual_calls': sum(a.fixture_calls for a in agents),
    }


case_rows, all_orders, all_trades, all_decisions, all_events = [], [], [], [], []
all_recorded_orders, all_recorded_trades = [], []
reference_calls = 0
for case in CASES:
    observed = run_case(case, True)
    reference = run_case(case, False)
    assert observed['normalized'] == reference['normalized'], (case['id'], 'observer changed market behavior')
    audit = observed['audit']
    all_orders.extend(observed['rows'])
    all_trades.extend(audit.trades)
    all_decisions.extend(audit.decisions)
    all_events.extend(audit.events)
    all_recorded_orders.extend(observed['recorder'].order_data)
    all_recorded_trades.extend(observed['recorder'].trade_data)
    reference_calls += reference['manual_calls']
    case_rows.append({
        'scenario_id': case['id'], 'scenario': case['title'], 'status': 'PASS',
        'orders': len(observed['rows']), 'trades': len(audit.trades),
        'executed_shares': sum(t['quantity'] for t in audit.trades),
        'event_count': len(audit.events), 'links_and_resources_checked': True,
        'observer_preserves_behavior': True,
    })
    print(case['id'] + ': PASS', flush=True)

assert not network_attempts
assert len({r['order_id'] for r in all_orders}) == len(all_orders)
assert len({r['decision_id'] for r in all_decisions}) == len(all_decisions)
assert len({r['trade_id'] for r in all_trades}) == len(all_trades)
write_csv(output / 'scenario_checks.csv', case_rows)
write_csv(output / 'all_order_lifecycle.csv', all_orders)
write_csv(output / 'all_trades_linked.csv', all_trades)
write_csv(output / 'all_recorded_orders.csv', all_recorded_orders)
write_csv(output / 'all_recorded_trades.csv', all_recorded_trades)
with (output / 'all_decisions.jsonl').open('w', encoding='utf-8') as stream:
    for row in all_decisions:
        stream.write(json.dumps(clean(row), ensure_ascii=False) + '\n')
with (output / 'all_order_events.jsonl').open('w', encoding='utf-8') as stream:
    for row in all_events:
        stream.write(json.dumps(clean(row), ensure_ascii=False) + '\n')
summary = {
    'source_commit': '074faff3c8f6255c917d2a2593a5b476c947ca84',
    'source_includes_step1_cap': True, 'scenario_count': len(case_rows),
    'scenario_checks_passed': len(case_rows), 'order_count': len(all_orders),
    'accepted_orders': sum(r['accepted'] for r in all_orders),
    'execution_status_counts': dict(Counter(r['execution_status'] for r in all_orders)),
    'trade_count': len(all_trades), 'executed_shares': sum(t['quantity'] for t in all_trades),
    'logged_manual_decisions': len(all_decisions), 'reference_manual_decisions': reference_calls,
    'event_count': len(all_events), 'model_api_calls': 0, 'network_attempts': len(network_attempts),
    'observer_behavior_comparisons_passed': len(case_rows),
    'scope': 'Manual-response offline engine cases; not new LLM market evidence and not reconstructed historical lifecycles',
    'output_directory': str(output),
}
(output / 'step2_summary.json').write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding='utf-8')
print('SUMMARY ' + json.dumps(summary, ensure_ascii=False), flush=True)
"""

if not LOGGED_ROOT.exists():
    shutil.copytree(CAPPED_ROOT, LOGGED_ROOT)
for relative_path, expected in EXPECTED_CAP_SHA256.items():
    text=(LOGGED_ROOT / relative_path).read_text(encoding="utf-8")
    if hashlib.sha256(text.encode("utf-8")).hexdigest() != expected:
        raise RuntimeError(f"Base source in the Step 2 copy contains other changes: {relative_path}. Preserve and inspect the changes first.")

new_files = {
    "src/order_lifecycle_audit.py": AUDIT_MODULE_SOURCE,
    "scripts/run_step2_lifecycle_offline.py": OFFLINE_RUNNER_SOURCE,
}
for relative_path, text in new_files.items():
    target = LOGGED_ROOT / relative_path
    if target.exists() and target.read_text(encoding="utf-8") != text:
        raise RuntimeError(f"Observer file contains other changes: {relative_path}; overwrite stopped.")
    ast.parse(text, filename=relative_path)
for relative_path, text in new_files.items():
    (LOGGED_ROOT / relative_path).write_text(text, encoding="utf-8")

print("Complete observer and offline runner are ready.")
print("Instrumented copy: ", LOGGED_ROOT)
print("Step 1 cap retained; the Step 1 copy remains available for the original comparison.")


## Code cell 5/8: Run 12 cases through the actual engine

Run each case once with the observer enabled and once with it disabled. Check that orders, trades, accounts, and prices agree between the two runs.

All agent responses are manually specified. Order creation, validation, resource handling, matching, execution transfers, and cancellations use the project source. The test subprocess blocks network connections.


In [ ]:
import uuid

RUN_OUTPUT_DIR = WORK_DIR / "results" / ("run_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S") + "_" + uuid.uuid4().hex[:6])
RUN_OUTPUT_DIR.mkdir(parents=True, exist_ok=False)
runner_path = LOGGED_ROOT / "scripts/run_step2_lifecycle_offline.py"
process = subprocess.run(
    [str(TEST_PYTHON), str(runner_path), "--source", str(LOGGED_ROOT), "--output", str(RUN_OUTPUT_DIR)],
    text=True, capture_output=True, cwd=LOGGED_ROOT, timeout=180,
)
if process.returncode:
    print(process.stdout)
    raise RuntimeError("Some checks failed. Retain the complete error:\n" + process.stderr[-10000:])
print("\n".join(line for line in process.stdout.splitlines() if not line.startswith("SUMMARY ")))
SUMMARY = json.loads((RUN_OUTPUT_DIR / "step2_summary.json").read_text(encoding="utf-8"))
print("\nCases passed: ", SUMMARY["scenario_checks_passed"], "/", SUMMARY["scenario_count"])
print("Model API calls: ", SUMMARY["model_api_calls"])
print("Result directory: ", RUN_OUTPUT_DIR)


## Code cell 6/8: Independently reconcile exported logs

Reload CSV/JSONL files, accumulate actual executions by both counterparty order IDs, and reconcile order status, the quantity ledger, and resource release. Aggregate counts cannot replace order-level linkage, so IDs and quantities are checked directly.

The quantity ledger is:

**Accepted quantity = executed quantity + current remaining quantity (including the remainder retained at cancellation) + quantity removed by non-trade operations after acceptance.**

For example, a market buy order is accepted for 1000 shares and executes 500. Of the remaining 500, 46 are removed during quote conversion, leaving 454 resting shares. The 46 removed shares are not executions.


In [ ]:
checks = read_csv(RUN_OUTPUT_DIR / "scenario_checks.csv")
orders = read_csv(RUN_OUTPUT_DIR / "all_order_lifecycle.csv")
trades = read_csv(RUN_OUTPUT_DIR / "all_trades_linked.csv")
recorded_orders = read_csv(RUN_OUTPUT_DIR / "all_recorded_orders.csv")
recorded_trades = read_csv(RUN_OUTPUT_DIR / "all_recorded_trades.csv")
decisions = read_jsonl(RUN_OUTPUT_DIR / "all_decisions.jsonl")
events = read_jsonl(RUN_OUTPUT_DIR / "all_order_events.jsonl")

assert len(checks) == 12 and all(row["status"] == "PASS" for row in checks)
assert all(row["observer_preserves_behavior"] == "True" for row in checks)
by_order = {row["order_id"]: row for row in orders}
decision_ids = {row["decision_id"] for row in decisions}
assert len(by_order) == len(orders) == 19
assert len(decision_ids) == len(decisions) == 32
assert len(trades) == 5 and sum(float(row["quantity"]) for row in trades) == 780

executed_by_order = defaultdict(float)
for trade in trades:
    for key in ("buyer_order_id", "seller_order_id"):
        oid = trade[key]
        assert oid in by_order and by_order[oid]["accepted"] == "True"
        executed_by_order[oid] += float(trade["quantity"])
for row in orders:
    assert row["decision_id"] in decision_ids
    assert abs(executed_by_order[row["order_id"]] - float(row["executed_quantity"])) < 1e-8
    if row["accepted"] == "True":
        residual = float(row["accepted_quantity"]) - float(row["executed_quantity"]) - float(row["remaining_quantity_raw"]) - float(row["post_acceptance_removed_quantity"])
        assert abs(residual) < 1e-8
    if row["final_state"] in ("filled", "cancelled"):
        assert abs(float(row["cash_commitment_final"])) < 1e-8
        assert abs(float(row["share_commitment_final"])) < 1e-8
for row in recorded_orders:
    assert row["order_id"] in by_order and row["decision_id"] == by_order[row["order_id"]]["decision_id"]
assert len(recorded_orders) == 16
assert {row["trade_id"] for row in recorded_trades} == {row["trade_id"] for row in trades}
assert all(row["buyer_order_id"] in by_order and row["seller_order_id"] in by_order for row in recorded_trades)

events_by_run = defaultdict(list)
for event in events:
    events_by_run[event["run_id"]].append(event)
    if event["event_type"] == "VALIDATION_REJECTED":
        assert event["resource_before"] == event["resource_after"]
for run_events in events_by_run.values():
    assert [row["event_seq"] for row in run_events] == list(range(1, len(run_events) + 1))
assert SUMMARY["model_api_calls"] == 0 and SUMMARY["network_attempts"] == 0

print("Independent reconciliation of exported logs: PASS.")
display_table(checks, [("scenario","Test case"),("status","Result"),("orders","Orders"),
                       ("trades","Trades"),("executed_shares","Executed shares")])
counts = Counter(row["execution_status"] for row in orders)
labels = {"fully_filled":"Fully filled","partially_filled":"Partially filled","unfilled":"Accepted but unfilled","rejected":"Validation rejection"}
display_table([{"status":labels[key],"count":counts[key]} for key in labels], [("status","Order classification"),("count","Orders")])
print("Linked 19 orders and 5 actual trades; the observed group contains 32 manual decisions, with another 32 in the observer-disabled control.")
print("Total events: ", len(events), "; model API calls: 0.")


## Code cell 7/8: Follow one order through its lifecycle

The default trace shows a buy order that executes 40 shares and then cancels its remaining 60. It displays validation, commitment, execution-related release, transfers, cancellation, and release of the remaining commitment.

**Resource release is distinct from an execution transfer.** The engine first releases commitments and then updates actual positions/cash; both actions are recorded separately.


In [ ]:
CASE_TO_INSPECT = "partial_then_cancel"
case_orders = read_csv(RUN_OUTPUT_DIR / CASE_TO_INSPECT / "order_lifecycle.csv")
target = next(row for row in case_orders if row["agent_id"] == "buyer")
case_events = read_jsonl(RUN_OUTPUT_DIR / CASE_TO_INSPECT / "order_events.jsonl")
selected_types = {"ORDER_REGISTERED","VALIDATION_STARTED","RESOURCE_COMMITTED","VALIDATION_ACCEPTED",
                  "TRADE_EXECUTION_STARTED","TRADE_EXECUTED","POSITION_UPDATED","CANCEL_STARTED",
                  "RESOURCE_RELEASED","CANCEL_COMPLETED","STATE_CHANGED"}
trace_rows=[]
for event in case_events:
    is_trade_event = event["event_type"] in {"TRADE_EXECUTION_STARTED", "TRADE_EXECUTED"}
    related = event["order_id"] == target["order_id"] or (
        is_trade_event and target["order_id"] in (event["buyer_order_id"], event["seller_order_id"]))
    if not related or event["event_type"] not in selected_types:
        continue
    snapshot=event.get("order") or {}
    resource=event.get("resource_after") or {}
    if "available_cash" not in resource:
        resource=resource.get(target["agent_id"],{})
    trace_rows.append({"sequence":event["event_seq"],"event":event["event_type"],
                       "owner":snapshot.get("agent_id", "Both counterparties" if is_trade_event else event.get("agent_id", "")),
                       "state":snapshot.get("state",""),"filled":snapshot.get("filled_quantity",""),
                       "remaining":snapshot.get("remaining_quantity",""),
                       "available_cash":resource.get("available_cash",""),
                       "committed_cash":resource.get("committed_cash","")})
assert all(row["owner"] in (target["agent_id"], "Both counterparties") for row in trace_rows)
display_table(trace_rows, [("sequence","Event sequence"),("owner","Actor"),("event","Event"),("state","Order state"),
                           ("filled","Cumulative execution"),("remaining","Remaining quantity"),
                           ("available_cash","Available cash"),("committed_cash","Committed cash")])
print("Order requested: ", target["requested_quantity"], "shares; executed: ", target["executed_quantity"], "shares.")
print("Execution status: ", target["execution_status"], "; final state: ", target["final_state"])
print("Cancelled remaining quantity: ", target["cancelled_remaining_quantity"], "; final committed cash: ", target["cash_commitment_final"])
print("\nQuantity-adjustment case: ")
for case in ("allocation_reduction", "market_resize"):
    rows=read_csv(RUN_OUTPUT_DIR / case / "order_lifecycle.csv")
    focus=next(row for row in rows if row["agent_id"] == ("seller" if case == "allocation_reduction" else "buyer"))
    print(case, "Requested", focus["requested_quantity"], "Accepted", focus["accepted_quantity"],
          "Executed", focus["executed_quantity"], "Remaining", focus["remaining_quantity_raw"],
          "Removed after acceptance", focus["post_acceptance_removed_quantity"])


## Code cell 8/8: Save the result archive

Create a ZIP containing the observer, offline runner, order-level results, event records, and the Step 1 patch if available. Package an explicit set of files rather than the entire Python environment.

The team can attach logging before a new single-stock simulation through `attach_to_simulation`. This interface is provided; a complete 15-round LLM integration run and recovery of historical missing order links are separate work.


In [ ]:
import zipfile

progress_text = (
    "This stage implemented and validated order-lifecycle logging offline. Decision, order, and trade IDs connect complete responses, validation, quantity adjustments, matching, cancellation/replacement, and resource changes. "
    "All 12 manual-response cases passed, with identical market outcomes when the observer was enabled or disabled. The observed group contains 19 proposed orders: 16 accepted and 3 rejected. "
    "Using actual executed quantities, 7 orders were fully filled, 3 partially filled, and 6 accepted but unfilled. There were 5 trades totaling 780 shares. "
    "Cancellation after partial execution and quantity reductions are recorded separately; partial borrow approval is not classified as execution. "
    "These results validate the observer mechanism, not new LLM behavior, and do not recover the complete missing order links in the four historical runs."
)
(RUN_OUTPUT_DIR / "progress_note_cn.txt").write_text(progress_text, encoding="utf-8")
with (RUN_OUTPUT_DIR / "historical_linkage_audit.csv").open("w",encoding="utf-8-sig",newline="") as stream:
    writer=csv.DictWriter(stream,fieldnames=list(legacy_rows[0]));writer.writeheader();writer.writerows(legacy_rows)

readme = """AI Herding Step 2 — offline order lifecycle audit
Source revision: 074faff3c8f6255c917d2a2593a5b476c947ca84, with the verified Step 1 cap.

New source files:
  src/order_lifecycle_audit.py
  scripts/run_step2_lifecycle_offline.py
The notebook creates a separate source_logged copy; it does not push changes.

Reproduction from a project root containing these files and the Step 1 cap:
  python scripts/run_step2_lifecycle_offline.py --source . --output NEW_EMPTY_OUTPUT_DIRECTORY
Use a fresh output directory. This runner uses manual agent responses and blocks network connections.

Main outputs:
  scenario_checks.csv               12 scenario checks and observer comparisons
  all_order_lifecycle.csv            decision/order linkage and execution/disposition classification
  all_trades_linked.csv              both order IDs and both decision IDs for every trade
  all_recorded_orders.csv            real DataRecorder rows with added IDs and quantities
  all_recorded_trades.csv            real DataRecorder rows with both counterparty order IDs
  all_decisions.jsonl                complete successful decision responses, including Add/Cancel/Replace
  all_order_events.jsonl             actual event order, resource snapshots and quantity changes
  CASE/order_snapshots.csv           all orders across rounds, including resting orders
  CASE/agent_snapshots.csv           available/committed cash and shares

Scope: single-stock manual-response cases, zero fees, no dividend or interest payments.
Historical runs lacking IDs remain incomplete. New LLM runs and 15-round integration are separate work.
"""
(RUN_OUTPUT_DIR / "README.txt").write_text(readme,encoding="utf-8")
ZIP_PATH = WORK_DIR / (RUN_OUTPUT_DIR.name + "_Step2_Results.zip")
with zipfile.ZipFile(ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(RUN_OUTPUT_DIR.rglob("*")):
        if path.is_file():
            archive.write(path,arcname="results/" + path.relative_to(RUN_OUTPUT_DIR).as_posix())
    for relative_path in new_files:
        archive.write(LOGGED_ROOT / relative_path,arcname=relative_path)
    step1_patch=STEP1_WORK / "results/step1_order_size_limit.patch"
    if step1_patch.exists():
        archive.write(step1_patch,arcname="step1_order_size_limit.patch")

print("Step 2 offline validation completed.")
print("Model API calls: 0.")
print("Reportable progress:\n",progress_text)
print("\nResult archive: ", ZIP_PATH)
print("Order-level table: ", RUN_OUTPUT_DIR / "all_order_lifecycle.csv")
print("Complete event stream: ", RUN_OUTPUT_DIR / "all_order_events.jsonl")
print("\nPress Ctrl + S to save this notebook, then export HTML to retain the local results.")


### Expected results and next steps

- All 12 cases show `PASS`; model API calls and test network requests are both 0.
- Of 19 orders, 7 are fully filled, 3 partially filled, 6 accepted but unfilled, and 3 rejected.
- There are 5 actual trades totaling 780 shares. Each trade fills both a buy and a sell order.
- These numbers come from the manually specified test inputs and validate the mechanism only.

Save the executed `.ipynb` and exported HTML for review. Step 3 provides replayable prompt comparisons; model API access is required when generating new model responses.
